# Model A Bakeoff — CatBoost vs XGBoost vs LightGBM

Comparison notebook (kept for future testing). The production winner is **CatBoost**,
trained by `train_model_a.py`. This notebook re-runs the full per-position, games-weighted,
walk-forward-tuned bakeoff against the 2025 holdout, plus a fair matched-subset baseline check.

Predicting `target_ppg` (half-PPR points per game). NaN priors are passed through natively.
2025 is never seen during tuning.

In [ ]:
import sys
import json
import time
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    # running as a notebook: __file__ is undefined, find the dir holding the dataset
    HERE = next((d for d in [Path.cwd(), Path.cwd()/'fantasy'/'seasonal_projections',
                             Path('fantasy/seasonal_projections')]
                 if (d/'season_dataset_2014_2025.csv').exists()), Path.cwd())
DATA      = HERE / "season_dataset_2014_2025.csv"
OUT_JSON  = HERE / "model_a_compare_results.json"
POSITIONS = ["QB", "RB", "WR", "TE"]
CV_VAL_SEASONS = [2021, 2022, 2023, 2024]   # walk-forward validation folds within train
HOLDOUT   = 2025
SEED      = 42

EXCLUDE = {
    "player_id", "player", "norm_name", "team", "position", "season", "reconstructed",
    "target_ppg", "target_games", "sample_weight",
    "adp_half_ppr", "adp_overall_rank", "adp_pos_rank", "sleeper_pts_half_ppr",
}


In [ ]:
def weighted_mae(y, p, w):
    return float(np.average(np.abs(np.asarray(y) - np.asarray(p)), weights=w))


# ── Hyperparameter grids (small, regularization-focused for a modest sample) ─
def xgb_grid():
    for md, lr, lam, mcw in itertools.product([3, 4], [0.03, 0.06], [1.0, 5.0], [5, 10]):
        yield dict(n_estimators=500, max_depth=md, learning_rate=lr, reg_lambda=lam,
                   min_child_weight=mcw, subsample=0.8, colsample_bytree=0.8,
                   reg_alpha=0.5, objective="reg:squarederror", random_state=SEED,
                   n_jobs=-1, verbosity=0)


def lgb_grid():
    for nl, lr, lam, mcs in itertools.product([15, 31], [0.03, 0.06], [1.0, 5.0], [20, 40]):
        yield dict(n_estimators=500, num_leaves=nl, learning_rate=lr, reg_lambda=lam,
                   min_child_samples=mcs, subsample=0.8, colsample_bytree=0.8,
                   max_depth=-1, random_state=SEED, n_jobs=-1, verbose=-1)


def cat_grid():
    for depth, lr, l2 in itertools.product([4, 6], [0.03, 0.06], [3.0, 6.0]):
        yield dict(iterations=500, depth=depth, learning_rate=lr, l2_leaf_reg=l2,
                   loss_function="MAE", random_seed=SEED, verbose=0, allow_writing_files=False)


def fit_predict(algo, params, Xtr, ytr, wtr, Xte):
    if algo == "xgboost":
        m = xgb.XGBRegressor(**params)
        m.fit(Xtr, ytr, sample_weight=wtr)
    elif algo == "lightgbm":
        m = lgb.LGBMRegressor(**params)
        m.fit(Xtr, ytr, sample_weight=wtr)
    else:  # catboost
        m = CatBoostRegressor(**params)
        m.fit(Xtr, ytr, sample_weight=wtr)
    return m, m.predict(Xte)


GRIDS = {"xgboost": xgb_grid, "lightgbm": lgb_grid, "catboost": cat_grid}


def tune(algo, pos_df, feats):
    """Walk-forward CV over CV_VAL_SEASONS; return best params by mean fold wMAE."""
    best, best_score = None, np.inf
    for params in GRIDS[algo]():
        fold_scores = []
        for val_season in CV_VAL_SEASONS:
            tr = pos_df[pos_df.season < val_season]
            va = pos_df[pos_df.season == val_season]
            if len(tr) < 50 or len(va) < 10:
                continue
            _, pred = fit_predict(algo, params, tr[feats], tr.target_ppg, tr.sample_weight, va[feats])
            fold_scores.append(weighted_mae(va.target_ppg, pred, va.sample_weight))
        if fold_scores:
            score = float(np.mean(fold_scores))
            if score < best_score:
                best, best_score = params, score
    return best, best_score


def main():
    df = pd.read_csv(DATA)
    df = df[df.target_ppg.notna()].copy()             # Model A usable rows only
    feats = [c for c in df.columns if c not in EXCLUDE]
    print(f"Features: {len(feats)}  | usable rows: {len(df):,}\n")

    results = {}
    for pos in POSITIONS:
        pos_df = df[df.position == pos].copy()
        train  = pos_df[pos_df.season <= 2024]
        hold   = pos_df[pos_df.season == HOLDOUT]
        results[pos] = {"n_train": len(train), "n_holdout": len(hold), "algos": {}, "baselines": {}}
        print(f"=== {pos}  (train={len(train)}  holdout={len(hold)}) ===")

        # baselines on holdout (only where the baseline column is present)
        for bname in ["prior_ppg", "ppg_3yr"]:
            sub = hold[hold[bname].notna()]
            if len(sub):
                results[pos]["baselines"][bname] = {
                    "wMAE": round(weighted_mae(sub.target_ppg, sub[bname], sub.sample_weight), 3),
                    "n": len(sub),
                }

        for algo in ["catboost", "xgboost", "lightgbm"]:
            t = time.time()
            best, cv_score = tune(algo, train, feats)
            _, pred = fit_predict(algo, best, train[feats], train.target_ppg,
                                  train.sample_weight, hold[feats])
            wmae = weighted_mae(hold.target_ppg, pred, hold.sample_weight)
            mae  = float(np.mean(np.abs(hold.target_ppg.values - pred)))
            rho  = float(spearmanr(pred, hold.target_ppg).statistic)
            results[pos]["algos"][algo] = {
                "cv_wMAE": round(cv_score, 3), "holdout_wMAE": round(wmae, 3),
                "holdout_MAE": round(mae, 3), "holdout_rho": round(rho, 3),
                "best_params": {k: v for k, v in best.items()
                                if k in ("max_depth", "depth", "num_leaves", "learning_rate",
                                         "reg_lambda", "l2_leaf_reg", "min_child_weight",
                                         "min_child_samples")},
            }
            print(f"  {algo:9}  cv_wMAE={cv_score:.3f}  holdout wMAE={wmae:.3f}  "
                  f"MAE={mae:.3f}  rho={rho:.3f}  ({time.time()-t:.0f}s)")
        print()

    OUT_JSON.write_text(json.dumps(results, indent=2))
    print(f"Wrote {OUT_JSON}\n")

    # ── Overall comparison (holdout rows pooled, games-weighted across positions) ─
    print("=" * 64)
    print("  OVERALL holdout wMAE by algorithm (lower = better)")
    print("=" * 64)
    print(f"  {'position':9} {'baseline*':>10} {'catboost':>9} {'xgboost':>9} {'lightgbm':>9}")
    for pos in POSITIONS:
        r = results[pos]
        base = min((b["wMAE"] for b in r["baselines"].values()), default=float("nan"))
        c = r["algos"]["catboost"]["holdout_wMAE"]
        x = r["algos"]["xgboost"]["holdout_wMAE"]
        l = r["algos"]["lightgbm"]["holdout_wMAE"]
        print(f"  {pos:9} {base:>10.3f} {c:>9.3f} {x:>9.3f} {l:>9.3f}")
    print("  * best of prior_ppg / ppg_3yr naive baselines")
    print("\n  Rank quality (Spearman rho, higher = better):")
    print(f"  {'position':9} {'catboost':>9} {'xgboost':>9} {'lightgbm':>9}")
    for pos in POSITIONS:
        a = results[pos]["algos"]
        print(f"  {pos:9} {a['catboost']['holdout_rho']:>9.3f} {a['xgboost']['holdout_rho']:>9.3f} {a['lightgbm']['holdout_rho']:>9.3f}")
    return results


## Run the bakeoff

In [ ]:
results = main()


## Fair model-vs-baseline check (matched rows, rookies excluded)

Scores each tuned model on the same holdout rows the naive baseline can predict,
so the comparison isn't penalizing the model for rookie rows the baseline skips.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, warnings
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostRegressor
warnings.filterwarnings("ignore")
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    # running as a notebook: __file__ is undefined, find the dir holding the dataset
    HERE = next((d for d in [Path.cwd(), Path.cwd()/'fantasy'/'seasonal_projections',
                             Path('fantasy/seasonal_projections')]
                 if (d/'season_dataset_2014_2025.csv').exists()), Path.cwd())
df = pd.read_csv(HERE / "season_dataset_2014_2025.csv")
df = df[df.target_ppg.notna()].copy()
res = json.loads((HERE / "model_a_compare_results.json").read_text())
EXCLUDE = {"player_id","player","norm_name","team","position","season","reconstructed",
           "target_ppg","target_games","sample_weight","adp_half_ppr","adp_overall_rank",
           "adp_pos_rank","sleeper_pts_half_ppr"}
feats = [c for c in df.columns if c not in EXCLUDE]
wmae = lambda y,p,w: float(np.average(np.abs(np.asarray(y)-np.asarray(p)), weights=w))

def full_params(algo, saved):
    if algo == "xgboost":
        return dict(n_estimators=500, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5,
                    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0, **saved)
    if algo == "lightgbm":
        return dict(n_estimators=500, subsample=0.8, colsample_bytree=0.8,
                    random_state=42, n_jobs=-1, verbose=-1, **saved)
    return dict(iterations=500, loss_function="MAE", random_seed=42, verbose=0,
                allow_writing_files=False, **saved)

def fit_pred(algo, params, Xtr, ytr, wtr, Xte):
    M = {"xgboost": xgb.XGBRegressor, "lightgbm": lgb.LGBMRegressor, "catboost": CatBoostRegressor}[algo]
    m = M(**params); m.fit(Xtr, ytr, sample_weight=wtr); return m.predict(Xte)

print(f"{'pos':4} {'algo':9} {'rows':>5} {'model wMAE':>11} {'ppg_3yr base':>13} {'model better by':>16}")
for pos in ["QB","RB","WR","TE"]:
    pos_df = df[df.position==pos]
    tr = pos_df[pos_df.season<=2024]
    ho = pos_df[(pos_df.season==2025) & pos_df.ppg_3yr.notna()]   # matched subset
    base = wmae(ho.target_ppg, ho.ppg_3yr, ho.sample_weight)
    for algo in ["catboost","xgboost","lightgbm"]:
        saved = res[pos]["algos"][algo]["best_params"]
        pred = fit_pred(algo, full_params(algo, saved), tr[feats], tr.target_ppg, tr.sample_weight, ho[feats])
        m = wmae(ho.target_ppg, pred, ho.sample_weight)
        print(f"{pos:4} {algo:9} {len(ho):>5} {m:>11.3f} {base:>13.3f} {base-m:>+16.3f}")
    print()
